### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="homeq_default_prediction",
    dataset_year="2016",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Other", # Website from a book
    original_dataset_source_download_link="http://www.creditriskanalytics.net/datasets-private2.html",
    download_description=r"""
We get the data from the original source, not from the Kaggle reupload (https://www.kaggle.com/datasets/ajay1735/hmeq-data).

wget http://www.creditriskanalytics.net/uploads/1/9/5/1/19511601/hmeq.csv \
&& mkdir -p local-data-warehouse/homeq_default_prediction \
&& mv hmeq.csv local-data-warehouse/homeq_default_prediction
""",
    # References
    academic_reference_bibtex=r"""@book{baesens2016credit,
  title={Credit risk analytics: Measurement techniques, applications, and examples in SAS},
  author={Baesens, Bart and Roesch, Daniel and Scheule, Harald},
  year={2016},
  publisher={John Wiley \& Sons}
}
""",
    academic_reference_bibtex_key="baesens2016credit",
    license="None", # copyright to the authors of the book most likely / website owner
    data_tags=["IID"],
    curation_comments="""
- We rename the target column and make its values more semantically meaningful (0 -> "no_default", 1 -> "default").
- We noticed several columns missing a large number of features. In all these cases the loan also did not have a value for "REASON". We drop 252 rows that dont have a reason for their loan in the data as these are most likely data errors that we want to ignore.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="DefaultStatus",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="DefaultStatus",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "hmeq.csv")

df = df[~df["REASON"].isna()]

as_cat_type = [
    "REASON", "JOB", "BAD",
]
df[as_cat_type] = df[as_cat_type].astype("category")

df["DefaultStatus"] = df["BAD"].map({0: "paid_loan", 1: "default_or_seriously_delinquent"})
df = df.drop(columns=["BAD"])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print("Loaded data shape:", df.shape)

Loaded data shape: (5708, 13)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 5,708
Columns: 13
Use sampling: False (sample size: 5,708)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['CLAGE', 'VALUE', 'MORTDUE', 'DEBTINC', 'LOAN', 'YOJ', 'CLNO', 'NINQ', 'DELINQ', 'DEROG']
Rows remaining as candidates after top-10 filter: 0 (of 5,708)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,LOAN,MORTDUE,VALUE,REASON,JOB,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC,DefaultStatus
0,12800,81030.0,100333.0,DebtCon,Mgr,11.0,0.0,1.0,80.148943,4.0,29.0,42.319370,default_or_seriously_delinquent
1,18800,88828.0,116683.0,DebtCon,ProfExe,6.0,0.0,0.0,141.841398,1.0,20.0,21.679278,paid_loan
2,28200,90485.0,123674.0,DebtCon,Other,1.0,0.0,1.0,189.734742,2.0,25.0,41.676986,paid_loan
3,16400,62786.0,79154.0,DebtCon,NaN,7.0,NaN,NaN,NaN,NaN,NaN,24.838940,paid_loan
4,14000,139000.0,159000.0,DebtCon,Mgr,13.0,0.0,1.0,171.433333,1.0,32.0,NaN,default_or_seriously_delinquent


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,JOB,category,172.0,3.01,6.0,"Other, ProfExe, Office, Mgr, Self, Sales"
1,REASON,category,0.0,0.00,2.0,"DebtCon, HomeImp"
2,DefaultStatus,category,0.0,0.00,2.0,"paid_loan, default_or_seriously_delinquent"
3,DEBTINC,float64,1211.0,21.22,4497.0,"37.1081, 42.3194, 21.6793, 41.677, 24.8389, 27.1482, 36.6112, 36.1123, 29.012, 38.5286"
4,DEROG,float64,614.0,10.76,11.0,"0.0, 1.0, 2.0, 3.0, 4.0, 6.0, 5.0, 7.0, 8.0, 10.0"
5,DELINQ,float64,484.0,8.48,14.0,"0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 11.0"
6,MORTDUE,float64,435.0,7.62,4903.0,"42000.0, 47000.0, 45000.0, 124000.0, 55000.0, 62000.0, 70000.0, 65000.0, 68000.0, 54000.0"
7,NINQ,float64,412.0,7.22,16.0,"0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 10.0, 8.0"
8,YOJ,float64,411.0,7.20,96.0,"0.0, 1.0, 2.0, 5.0, 4.0, 6.0, 3.0, 9.0, 10.0, 8.0"
9,CLAGE,float64,202.0,3.54,5189.0,"206.9667, 102.5, 123.7667, 177.5, 95.3667, 117.6667, 189.7, 179.5667, 109.5667, 97.4"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
LOAN,5708.0,18722.319552,11327.206630,1100.000000,89900.000000
MORTDUE,5273.0,73950.002008,44543.595404,2063.000000,399550.000000
VALUE,5616.0,102017.995021,57798.044524,8000.000000,855909.000000
YOJ,5297.0,8.892458,7.517474,0.000000,41.000000
DEROG,5094.0,0.249509,0.830154,0.000000,10.000000
DELINQ,5224.0,0.440084,1.113099,0.000000,15.000000
CLAGE,5506.0,178.953949,85.710420,0.000000,1168.233561
NINQ,5296.0,1.194298,1.728890,0.000000,17.000000
CLNO,5571.0,21.363130,10.117606,0.000000,71.000000
DEBTINC,4497.0,33.995624,8.248962,0.524499,203.312149


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column        rank                                               
DefaultStatus 1                           paid_loan   4567  80.01
              2     default_or_seriously_delinquent   1141  19.99
JOB           1                               Other   2320  40.64
              2                             ProfExe   1252  21.93
              3                              Office    921  16.14
              4                                 Mgr    746  13.07
              5                                Self    188   3.29
REASON        1                             DebtCon   3928  68.82
              2                             HomeImp   1780  31.18

In [8]:
# Target Distribution
target_df

,count,pct
DefaultStatus,,
paid_loan,4567,80.01
default_or_seriously_delinquent,1141,19.99


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to homeq_default_prediction/019d6daf-af36-77f3-9486-588def3d195f
019d6daf-af36-77f3-9486-588def3d195f
fe9846eb10af3376d7efaf53f07a6a55217cb74e4986972204d80450c33a826e
